This is a attempt to train a predictive control using reinforcement learning.

## Setup Overview

### Agent: 

The point absorber. It records its motion (displacements) and takes actions (controls the dampning control).

### Environment: 

This is the cummins equation simulator with the randomly generated jonswap sea state

### Rewards: 

The mean power absorbed over medium time period 

Requirements: run from wave_rl environment as capytaine_env was using a to modern python for stable baselines

In [1]:
import os # for saving model
import gymnasium # going to customise a openai gym environment
from gymnasium import Env 
from gymnasium.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete
from stable_baselines3 import PPO # built in training model
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

In [2]:
import numpy as np

In [ ]:
from Functions import generate_frequencies, jonswap_frequency_amplitudes, generate_buoy, plot_history, solve_with_capytaine, get_cummins_components, solve_cummins_stepwise_latch, solve_cummins_stepwise_no_latch, calc_power_absorbed, plot_power

Model Parameters:

In [ ]:
# --- Default run configuration (override via CLI) ---
SAVE = False
TSPAN = 120
SEED = 42
NFREQ = 50
PEAK_PERIOD = 8.0
HS = 2.0
BUOY_MASS = 1000.0
BUOY_RADIUS = 2.0
WATER_DENSITY = 1025.0
WATER_DEPTH = np.inf  # or a finite depth if you need it
WAVE_DIRECTION = 0.0
WAVE_AMPLITUDE = None  # only if you still use it somewhere
C_PTO = 1e5
K_PTO = 0.0
RL_STEP_SIZE = 0.25
SOLVER_STEP_SIZE = 0.01

Modified Functions For RL

In [ ]:
# solve with capytaine once as this is the slow part

buoy = generate_buoy(radius=BUOY_RADIUS, mass=BUOY_MASS)

omegas, delta_omega = generate_frequencies(N=NFREQ, Tp=PEAK_PERIOD)

capytaine_dataset = solve_with_capytaine(body=buoy, omegas=omegas, wave_direction=WAVE_DIRECTION, water_depth=WATER_DEPTH, water_density=WATER_DENSITY)

wave_amplitudes = jonswap_frequency_amplitudes(omegas, delta_omega, Hs = 2.0, Tp=12.0 , gamma=3.3)



In [ ]:
# create the custom environment
class WaveEnv(Env):
    def __init__(self):
        self.action_space = Box(low=0, high=1e5, shape=(1,))
        self.observation_space = Box(low=-10, high=10, shape=(2,)) # do i need to change shape to track displacement over time
        self.state = (0,0)
        self.history = {'t': [0.0], 'x': [0.0], 'v': [0.0], 'F_ex': [F_ex_time(0.0)], 'c_pto': [C_PTO]}
        
        # random seed for each env

        A_heave_inf, t_kernel, kernel, K_heave, F_ex_time, F_ex_time_dot = get_cummins_components(body=buoy, capytaine_dataset=capytaine_dataset, wave_direction=WAVE_DIRECTION, wave_amplitudes=wave_amplitudes, omegas=omegas, seed=SEED)

        # parameters 
        self.buoy = buoy
        self.omegas = omegas
        self.delta_omega = delta_omega
        self.wave_amplitudes = wave_amplitudes
        self.A_heave_inf = A_heave_inf
        self.t_kernel = t_kernel
        self.kernel = kernel
        self.K_heave = K_heave
        self.rl_step_size = RL_STEP_SIZE
        self.solver_step_size = SOLVER_STEP_SIZE
        self.simulation_length = 180

    def step(self, action): # this is for one simulation step action is the dampning

        # compute the simulation to update the state using the action
        self.history = solve_cummins_stepwise_no_latch(body=self.buoy, A_heave_inf=self.A_heave_inf, t_kernel=self.t_kernel, kernel=self.kernel, K_heave=self.K_heave, F_ex_time=self.F_ex_time, F_ex_time_dot=self.F_ex_time_dot, C_pto=self.action, K_pto=self.K_pto, t_span=[0,self.rl_step_size], dt=self.solver_step_size)
        self.state = self.history['x'][-20:], self.history['v'][-20:]

        # reduce the time remaining by 1
        self.simulation_length -= self.step_size
    
        # calculate the reward # just power absorbed over the last 20 steps
        self.reward = np.sum(np.array(self.state[1]) ** 2 * np.array(self.history['c_pto'][-20:]))

        # check if the episode is done
        self.done = self.simulation_length <= 0

        return self.state, self.reward, self.done, {}

    def reset(self):
        self.state = (0,0)
        self.simulation_length = 180
        return self.state

    
